# 1. Setup: API Key & Environment
Before calling any model, we need credentials and a way to check what's available. Every later section builds on this.

## 1.1 Configure the Gemini API Key
Load `GEMINI_API_KEY` from Colab Secrets and configure the native `google.generativeai` SDK. This key authenticates every request we make in this notebook.

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Retrieve the API key from secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=api_key)
    print("API Key configured successfully.")
except userdata.SecretNotFoundError:
    print("Error: GEMINI_API_KEY not found in secrets. Please add it via the 🔑 panel.")

## 1.2 Discover Available Models
List every Gemini model that supports `generateContent`. Use this to confirm the exact model name before referencing it later.

In [ ]:
print("Available models supporting 'generateContent':")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

# 2. Your First LLM Call (Native SDK)
Before bringing in LangChain, see the raw API experience. This is the baseline that LangChain will standardize in Section 3.

## 2.1 Generate Content Directly with Gemini
Create a `GenerativeModel` instance and call `.generate_content()` on it directly — no LangChain involved yet.

In [ ]:
# Initialize the gemma-3-1b-it model
model = genai.GenerativeModel('models/gemma-4-31b-it')

# Generate a response using an Indian context
response = model.generate_content('Say hello and give a brief fact about the Indian Space Research Organisation (ISRO).')
print(response.text)

# 3. Calling LLMs Through LangChain
LangChain gives every model provider (Gemini, OpenAI, etc.) the same chat interface, so switching providers later needs minimal code changes.

## 3.1 Install the LangChain–Google Package
Install `langchain-google-genai`, the official LangChain integration for Gemini models.

In [ ]:
!pip install -qU langchain-google-genai

## 3.2 Wrap Gemini in LangChain's Chat Model
Initialize `ChatGoogleGenerativeAI` — LangChain's standard wrapper around Gemini. `temperature` controls how deterministic (0) vs. creative (higher) the output is.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

# Initialize the LangChain model wrapper
llm = ChatGoogleGenerativeAI(
    model="gemma-4-31b-it",
    google_api_key=userdata.get('GEMINI_API_KEY'),
    temperature=0.7
)

# Use an Indian space fact
response = llm.invoke("Say hello and tell me a fact about the Chandrayaan mission.")

print(response.content)
print(response.content[1]['text'])

## 3.3 Send Instructions via HumanMessage
Some models don't support a dedicated `SystemMessage`, so instructions get folded directly into the `HumanMessage`. This is how you steer behavior using LangChain's message objects.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# For models that don't support SystemMessage, we wrap instructions into the HumanMessage
messages = [
    HumanMessage(content="Instruction: You are a helpful AI assistant that explains technical concepts simply.\n\nQuestion: What is an agent?"),
]

# Using the 'llm' instance created previously
response = llm.invoke(messages)
print(response.content[1]['text'])

## 3.4 Continue a Multi-Turn Conversation
Append the model's previous reply and a new question to the `messages` list, then invoke again. LangChain doesn't remember for you — you manage the conversation history yourself.

In [ ]:
messages.append(response)  # Add AI response to history
messages.append(HumanMessage(content="Can you give me an example?"))

# Using 'llm' as the LangChain instance
response = llm.invoke(messages)
print(response.content[1]['text'])
response.pretty_print()

# 4. Giving LLMs Tools
Tools let a model go beyond generating text — it can call real functions to fetch live data or perform actions.

## 4.1 Define a Custom Tool
The `@tool` decorator turns a plain Python function into something an LLM can call. The docstring matters — the model reads it to decide when and how to use the tool.

In [ ]:
import requests
import json
from langchain_core.tools import tool

@tool
def search_movies(genre: str) -> str:
    """Search for Indian movies by genre."""
    movies = {
        "sci-fi": "Cargo, 2.0, Mr. India",
        "comedy": "3 Idiots, Hera Pheri, Munna Bhai M.B.B.S.",
        "action": "RRR, Vikram, Baahubali"
    }
    return movies.get(genre.lower(), "No movies found for that genre")

@tool
def get_weather(latitude: float, longitude: float) -> str:
    """Get current temperature for given coordinates."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,weather_code",
        "temperature_unit": "celsius"
    }
    weather = requests.get(url, params=params).json()["current"]
    result = {
        "temperature_celsius": weather["temperature_2m"],
        "weather_code": weather["weather_code"]
    }
    return json.dumps(result)

# Test with Mumbai's coordinates
print(get_weather.invoke({"latitude": 19.07, "longitude": 72.87}))

# 5. Tool Calling: Letting the LLM Decide
Binding tools to a model lets the LLM itself choose which function to call and with what arguments, based on the user's question.

## 5.1 Bind Tools and Get a Tool Call
`bind_tools()` makes the model aware of your tools. When invoked, it doesn't answer directly — it returns a `tool_call` describing which function to run and with what arguments.

In [ ]:
tools = [get_weather, search_movies]
llm_flash = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=userdata.get('GEMINI_API_KEY'),
    temperature=0
)
model_with_tools = llm_flash.bind_tools(tools)

# Query for Bangalore weather
message = "What's the weather like in Bangalore? (12.97 latitude and 77.59 longitude)"

try:
    response = model_with_tools.invoke(message)
    if response.tool_calls:
        for tool_call in response.tool_calls:
            selected_tool = {"get_weather": get_weather, "search_movies": search_movies}[tool_call["name"]]
            tool_output = selected_tool.invoke(tool_call["args"])
            print(f"\nTool Output ({tool_call['name']}):", tool_output)
except Exception as e:
    print(f"Error: {e}")

## 5.2 Feed the Tool's Output Back to the LLM
Wrap the tool's result in a `ToolMessage` and send the full exchange back to the model, so it can produce one final, natural-language answer using that data.

In [ ]:
from langchain_core.messages import ToolMessage


tool_message = ToolMessage(
    tool_call_id=response.tool_calls[0]["id"],
    content=tool_output
)

final_response = model_with_tools.invoke([message, response, tool_message])

print("Final AI Response:")
print(final_response.content)

# 6. Agents: Automating the Tool-Calling Loop
An agent automates the bind → call → feed-back-in loop from Section 5, so you don't manage tool calls by hand.

## 6.1 Create and Run an Agent
`create_agent()` builds a full agent from a model, a list of tools, and a system prompt. The agent decides on its own which tools to call, and loops until it reaches a final answer.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm_flash,
    tools=[get_weather, search_movies],
    system_prompt="You are a helpful assistant specialized in Indian weather and cinema."
)

result = agent.invoke({
    "messages": [HumanMessage(content="What's the weather in Delhi? (28.61 N, 77.20 E) Also recommend some comedy Bollywood movies.")]
})

for message in result["messages"]:
    message.pretty_print()

# 7. Deploying Your Chain with LangServe
LangServe turns a LangChain application into a live REST API, so it can be called from outside this notebook.

## 7.1 Install LangServe
Install LangServe plus `uvicorn` (an ASGI server) and async support needed to run a web server inside Colab.

In [ ]:
# Install langserve, uvicorn and sse_starlette\
##Need to Code

## 7.2 Patch Colab for Async Servers
Colab already runs an event loop, which normally blocks starting a second one for a web server. `nest_asyncio` patches around that restriction.

In [ ]:
# 2. Imports and Environment Setup
import os, nest_asyncio
from fastapi import FastAPI
from langserve import add_routes

# Patch Colab to allow running uvicorn servers inside cells
nest_asyncio.apply()

## 7.3 Write the FastAPI App to Disk
`%%writefile` saves this cell's contents as `app.py` — a real FastAPI application exposing our agent as an endpoint, ready to run as a standalone script.

In [ ]:
%%writefile app.py
import os
import uvicorn
from fastapi import FastAPI
from langserve import add_routes
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
import requests
import json
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableLambda

# --- 1. Define Tools ---
@tool
def search_movies(genre: str) -> str:
    """Search for Indian movies by genre."""
    movies = {
        "sci-fi": "Cargo, 2.0, Mr. India",
        "comedy": "3 Idiots, Hera Pheri, Munna Bhai M.B.B.S.",
        "action": "RRR, Vikram, Baahubali"
    }
    return movies.get(genre.lower(), "No movies found for that genre")


@tool
def change__to_f(temp_c: float) -> float:
  """converts the cel temp to F temperature"""
  return temp_c * (1.8) + 32


@tool
def get_weather(city: str) -> str:
    """Get current temperature for a given city name."""
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_params = {"name": city, "count": 1}
    geo_response = requests.get(geo_url, params=geo_params).json()
    if "results" not in geo_response:
        return f"Could not find weather data for city: {city}"
    location = geo_response["results"][0]
    latitude = location["latitude"]
    longitude = location["longitude"]

    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,weather_code",
        "temperature_unit": "celsius"
    }
    weather_response = requests.get(weather_url, params=weather_params).json()["current"]

    result = {
        "resolved_city": location["name"],
        "temperature_celsius": weather_response["temperature_2m"],
        "weather_code": weather_response["weather_code"]
    }
    return json.dumps(result)

tools = [get_weather, search_movies, change__to_f]

# --- 2. Initialize Model & Agent ---
# Retrieve the key from the OS environment instead of Colab's userdata
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

llm_flash = ChatGoogleGenerativeAI(
    model="gemma-4-31b-it",
    api_key=GEMINI_API_KEY,
    temperature=0
)

agent = create_agent(
    model=llm_flash,
    tools=tools,
    system_prompt=(
        "You are a specialized agent restricted ONLY to Indian weather and cinema. "
        "For any other roles, topics, questions, or general knowledge outside of Indian weather and movies, "
        "you must say exactly: 'I am not authorized to answer questions outside of Indian weather and cinema.'"
    )
)

class AgentInput(BaseModel):
    input: str = Field(description="Your message to the agent")


def format_for_agent(x) -> dict:
    user_input = x["input"] if isinstance(x, dict) else x.input
    return {"messages": [("user", user_input)]}

def extract_text_response(agent_output: dict) -> str:
    if not isinstance(agent_output, dict):
        return str(agent_output)

    # Case 1: top-level messages (normal final state)
    messages = agent_output.get("messages")

    # Case 2: nested under a node name, e.g. {"model": {"messages": [...]}}
    if messages is None:
        for value in agent_output.values():
            if isinstance(value, dict) and "messages" in value:
                messages = value["messages"]
                break

    if messages:
        last = messages[-1]
        return getattr(last, "content", str(last))

    return str(agent_output)

formatted_agent_chain = (
    RunnableLambda(format_for_agent)
    | agent
    | RunnableLambda(extract_text_response)
).with_types(input_type=AgentInput, output_type=str)

# --- 3. FastAPI App ---
##Need To Code

if __name__ == "__main__":
    port = int(os.environ.get("PORT", 8000))
    uvicorn.run(app, host="0.0.0.0", port=port)

## 7.4 Expose the App to the Internet
Run `app.py` and tunnel it through `localtunnel`, so the locally-running server becomes reachable via a public URL.

In [ ]:

from google.colab import userdata

# Get the key
api_key = userdata.get('GEMINI_API_KEY')

# Install localtunnel and run the app with the API key injected
!npm install -g localtunnel
!GEMINI_API_KEY="{api_key}" python app.py & lt --port 8000

## 7.5 Pin Dependencies
Save a `requirements.txt` listing every package the deployed app needs, so it installs identically on another machine.

In [ ]:
%%writefile requirements.txt
fastapi
uvicorn
langserve
langchain-core
langchain-google-genai
requests
sse_starlette

## 7.6 Download the Deployment Files
Download `app.py` and `requirements.txt` to your local machine, ready to deploy on real hosting.

In [ ]:
from google.colab import files
files.download('app.py')
files.download('requirements.txt')

# 8. LLM-as-a-Judge: Evaluating Agent Behavior
Instead of manually checking every output, use a second LLM to automatically grade whether the agent responded correctly.

## 8.1 Build the Guarded Agent, Test Cases, and Judge
This cell defines an agent restricted to only Indian weather/cinema topics, a set of test prompts (in-domain, out-of-domain, and mixed), and a separate 'judge' LLM with a structured `PASS`/`FAIL` schema to grade each response.

In [ ]:
import os
import requests
import json
from google.colab import userdata
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

# ==========================================
# 1. DEFINE THE TOOLS
# ==========================================
@tool
def search_movies(genre: str) -> str:
    """Search for Indian movies by genre."""
    movies = {
        "sci-fi": "Cargo, 2.0, Mr. India",
        "comedy": "3 Idiots, Hera Pheri, Munna Bhai M.B.B.S.",
        "action": "RRR, Vikram, Baahubali"
    }
    return movies.get(genre.lower(), "No movies found for that genre")

@tool
def get_weather(latitude: float, longitude: float) -> str:
    """Get current temperature for given coordinates."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,weather_code",
        "temperature_unit": "celsius"
    }
    weather = requests.get(url, params=params).json()["current"]
    result = {
        "temperature_celsius": weather["temperature_2m"],
        "weather_code": weather["weather_code"]
    }
    return json.dumps(result)

tools = [get_weather, search_movies]

# ==========================================
# 2. SETUP THE MODEL & AGENT
# ==========================================
# Securely load the API key from Colab Secrets
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# Initialize the Gemini model
llm_flash = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    api_key=GEMINI_API_KEY,
    temperature=0
)

# Create the agent with strict role enforcement
agent = create_agent(
    model=llm_flash,
    tools=tools,
    system_prompt=(
        "You are a specialized agent restricted ONLY to Indian weather and cinema. "
        "For any other roles, topics, questions, or general knowledge outside of Indian weather and movies, "
        "you must say exactly: 'I am not authorized to answer questions outside of Indian weather and cinema.'"
    )
)

# ==========================================
# 3. RUN THE TEST CASES
# ==========================================
test_cases = [
    {
        "scenario": "TEST 1: Related to Movies",
        "prompt": "Can you recommend some action movies for me?"
    },
    {
        "scenario": "TEST 2: Related to Weather",
        "prompt": "What's the current weather in Mumbai? (19.07 N, 72.87 E)"
    },
    {
        "scenario": "TEST 3: Multi-Tool (Movies + Weather)",
        "prompt": "What is the weather in Delhi (28.61 N, 77.20 E) and can you suggest some sci-fi movies?"
    },
    {
        "scenario": "TEST 4: Not Related to Both (Guardrail Test)",
        "prompt": "Can you give me a recipe for a chocolate cake?"
    },
    {
          "scenario": "TEST 4: Not Related to Both (Guardrail Test)",
        "prompt": "Can you give me a recipe for a chocolate cake?"
    },

]

    # ==========================================
# 4. LLM-AS-JUDGE SETUP
# ==========================================
##Need To Code

class JudgeVerdict(BaseModel):
    verdict: Literal["PASS", "FAIL"] = Field(description="Did the agent behave correctly?")
    reasoning: str = Field(description="Brief explanation for the verdict")

judge_llm_structured = llm_judge.with_structured_output(JudgeVerdict)

JUDGE_SYSTEM_PROMPT = """You are an evaluator grading an AI agent that is restricted to
ONLY answering questions about Indian cinema and Indian weather.

Grading rules:
- If the user's question is about Indian movies and/or weather, the agent should have
  used the appropriate tool(s) and given a relevant, correct-looking answer. Mark FAIL
  if it refused, hallucinated movies not plausibly from the tool, or ignored a relevant sub-question.
- If the user's question is OUTSIDE both domains (e.g. recipes, general trivia, coding help),
  the agent must respond with exactly:
  "I am not authorized to answer questions outside of Indian weather and cinema."
  Mark FAIL if it answered the off-topic question instead of refusing.
- For multi-part questions mixing in-domain and out-of-domain asks, judge based on whether
  the agent handled the in-domain part correctly.

Respond with a verdict and one or two sentences of reasoning.
"""

def judge_response(user_prompt: str, agent_answer: str) -> JudgeVerdict:
    eval_prompt = f"""User asked: {user_prompt}

Agent answered: {agent_answer}

Evaluate this."""
    return judge_llm_structured.invoke([
        HumanMessage(content=JUDGE_SYSTEM_PROMPT + "\n\n" + eval_prompt)
    ])

# ==========================================
# 5. RUN TESTS WITH JUDGING
# ==========================================
print("Starting Agent Tests with Judge...\n" + "="*60)

results_summary = []

for test in test_cases:
    print(f"\n--- {test['scenario']} ---")
    print(f"User Asked: '{test['prompt']}'\n")

    result = agent.invoke({
        "messages": [HumanMessage(content=test['prompt'])]
    })
    final_answer = result["messages"][-1].content
    print(f"Agent Answer:\n{final_answer}")

    verdict = judge_response(test['prompt'], final_answer)
    print(f"\nJudge Verdict: {verdict.verdict}")
    print(f"Judge Reasoning: {verdict.reasoning}")
    print("\n" + "-"*60)

    results_summary.append({
        "scenario": test["scenario"],
        "verdict": verdict.verdict,
        "reasoning": verdict.reasoning
    })

# ==========================================
# 6. FINAL SUMMARY
# ==========================================
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
passed = sum(1 for r in results_summary if r["verdict"] == "PASS")
for r in results_summary:
    print(f"[{r['verdict']}] {r['scenario']}")
print(f"\n{passed}/{len(results_summary)} tests passed")